# 98 · Restauración de las tablas Ppto27

Sirve para **volver las tablas a un momento anterior**, por ejemplo después de ensayar la demo.

| Paso | Qué hace | Cuándo |
|---|---|---|
| **1 · Guardar punto** | Anota la versión de cada tabla `ppto27_*` con un nombre | **Antes** de cargar algo de prueba |
| **2 · Puntos guardados** | Lista los puntos disponibles | Cuando se quiera volver |
| **3 · Vista previa** | Muestra qué tablas y qué cargas cambiarían. No modifica nada | Antes de restaurar |
| **4 · Restaurar** | Devuelve cada tabla a la versión del punto | Solo con `CONFIRMAR = 'SI'` |
| **5 · Limpiar archivos** | Borra del Volume las carpetas de prueba (opcional) | Solo con `CONFIRMAR_BORRADO = 'SI'` |
| **6 · Estado final** | Muestra qué cargas quedaron en las tablas y en `10_cargas` | Al terminar |

Restaurar es necesario para repetir una prueba: las tablas recuerdan cada archivo por su huella y, si se vuelve a subir el mismo Excel sin restaurar, se marca como **REPETIDO** y no se vuelve a leer.

In [ ]:
import json, os, shutil
import pandas as pd
from pyspark.sql import functions as F

CAT = 'catalogo_ppto.esquema_ppto'
VOL = '/Volumes/catalogo_ppto/esquema_ppto/vol_administrado/ppto27'
PUNTOS = f'{VOL}/41_puntos_restauracion'
CARGAS = f'{VOL}/10_cargas'


def tablas():
    return sorted(r.tableName for r in spark.sql(f'SHOW TABLES IN {CAT}').collect() if r.tableName.startswith('ppto27_'))


def historia(t):
    return spark.sql(f'DESCRIBE HISTORY {CAT}.{t}').select('version', 'timestamp', 'operation')


def version_actual(t):
    return historia(t).agg(F.max('version')).first()[0]


def cargas_en_tablas(version=None):
    v = '' if version is None else f' VERSION AS OF {version}'
    try:
        return sorted(r.carga_id for r in spark.sql(f'SELECT DISTINCT carga_id FROM {CAT}.ppto27_brz_archivos{v}').collect())
    except Exception:
        return []


def ahora_utc():
    return str(spark.sql('SELECT current_timestamp() AS t').first().t)[:19]


print(f'{len(tablas())} tablas ppto27_* en {CAT} · hora actual (UTC) {ahora_utc()}')
print('Cargas que hoy están en las tablas:', cargas_en_tablas())

## 1 · Guardar punto de regreso
Correr **antes** de cargar algo de prueba. Cambiar el nombre si se guarda más de un punto.

In [ ]:
NOMBRE = 'antes_ensayo_demo'                       # sin espacios
NOTA = 'Antes de ensayar la demo de 3 plantillas'
SOBRESCRIBIR = False                                # True para reemplazar un punto con el mismo nombre

ruta = f'{PUNTOS}/{NOMBRE}.json'
if os.path.exists(ruta) and not SOBRESCRIBIR:
    print(f'Ya existe el punto "{NOMBRE}". Usar otro nombre o SOBRESCRIBIR = True.')
else:
    os.makedirs(PUNTOS, exist_ok=True)
    punto = dict(nombre=NOMBRE, nota=NOTA, fecha_utc=ahora_utc(),
                 versiones={t: int(version_actual(t)) for t in tablas()}, cargas=cargas_en_tablas())
    with open(ruta, 'w') as f:
        json.dump(punto, f, indent=1)
    print(f'Punto "{NOMBRE}" guardado · {punto["fecha_utc"]} UTC · {len(punto["versiones"])} tablas')
    print('Cargas en ese momento:', punto['cargas'])

## 2 · Puntos guardados

In [ ]:
filas = []
if os.path.isdir(PUNTOS):
    for a in sorted(os.listdir(PUNTOS)):
        if a.endswith('.json'):
            with open(f'{PUNTOS}/{a}') as f:
                p = json.load(f)
            filas.append(dict(punto=p['nombre'], fecha_utc=p['fecha_utc'], nota=p.get('nota', ''),
                              tablas=len(p['versiones']), cargas=len(p.get('cargas', []))))
if filas:
    display(pd.DataFrame(filas).sort_values('fecha_utc', ascending=False))
else:
    print('Todavía no hay puntos guardados (paso 1).')

## 3 · Vista previa (no modifica nada)
`PUNTO` es el nombre de un punto guardado. También acepta una hora UTC `'AAAA-MM-DD HH:MM:SS'` para volver a un momento en el que no se guardó un punto.

In [ ]:
PUNTO = 'antes_ensayo_demo'


def versiones_del_punto(p):
    ruta = f'{PUNTOS}/{p}.json'
    if os.path.exists(ruta):
        with open(ruta) as f:
            return json.load(f)['versiones']
    vers = {}
    for t in tablas():                                  # por hora: la última versión escrita hasta esa hora
        v = historia(t).where(F.col('timestamp') <= F.lit(p).cast('timestamp')).agg(F.max('version')).first()[0]
        vers[t] = None if v is None else int(v)
    return vers


vers = versiones_del_punto(PUNTO)
plan = []
for t in tablas():
    actual, objetivo = int(version_actual(t)), vers.get(t)
    if objetivo is None:
        accion = 'se deja igual (no existía en ese punto)'
    elif actual == objetivo:
        accion = 'sin cambios'
    else:
        accion = 'RESTAURAR'
    plan.append(dict(tabla=t, version_actual=actual, version_punto=objetivo, accion=accion))
PLAN = pd.DataFrame(plan)
PLAN['version_punto'] = PLAN.version_punto.astype('Int64')
PLAN_DE = PUNTO
display(PLAN)
v_arch = vers.get('ppto27_brz_archivos')
antes, hoy = set(cargas_en_tablas(v_arch)) if v_arch is not None else set(), set(cargas_en_tablas())
print(f'Tablas a restaurar: {(PLAN.accion == "RESTAURAR").sum()} de {len(PLAN)}')
print('Cargas que salen de las tablas:', sorted(hoy - antes) or 'ninguna')
print('Cargas que vuelven:', sorted(antes - hoy) or 'ninguna')

## 4 · Restaurar
Revisar la vista previa y escribir `CONFIRMAR = 'SI'`. Al terminar, volver a dejarlo en `'NO'`.

In [ ]:
CONFIRMAR = 'NO'

if CONFIRMAR != 'SI':
    print("No se hizo nada. Para restaurar escribir CONFIRMAR = 'SI' y volver a correr esta celda.")
elif 'PLAN' not in globals() or PLAN_DE != PUNTO:
    print('Primero correr la vista previa (paso 3) con este PUNTO.')
else:
    res = []
    for r in PLAN.itertuples():
        if r.accion != 'RESTAURAR':
            res.append((r.tabla, r.accion))
            continue
        try:
            spark.sql(f'RESTORE TABLE {CAT}.{r.tabla} TO VERSION AS OF {int(r.version_punto)}')
            res.append((r.tabla, f'restaurada a la versión {int(r.version_punto)}'))
        except Exception as e:
            res.append((r.tabla, 'ERROR: ' + str(e)[:150]))
    R = pd.DataFrame(res, columns=['tabla', 'resultado'])
    display(R)
    errores = R.resultado.str.startswith('ERROR').sum()
    print('Listo: tablas restauradas.' if not errores else f'{errores} tabla(s) con error: revisar arriba.')
    print('Cargas que quedan en las tablas:', cargas_en_tablas())

## 5 · Limpiar archivos de prueba del Volume (opcional)
Borra carpetas de `10_cargas` y, si se quiere, `40_tiempos` (tiempos y resúmenes del `08_demo`). Primero muestra lo que encontró.

In [ ]:
BORRAR = ['20261016_0900_Demo1_Imputacion_directa', '20261016_1000_Demo2_Soporte_directo', '20261016_1100_Demo3_Indirecto',
          '20261017_0800_Demo_Limpias', '20261017_0900_Demo_Con_errores']   # carpetas de demos anteriores en 10_cargas
BORRAR_TIEMPOS = True
CONFIRMAR_BORRADO = 'NO'

objetivos = [f'{CARGAS}/{c}' for c in BORRAR if c and '/' not in c and os.path.isdir(f'{CARGAS}/{c}')]
if BORRAR_TIEMPOS and os.path.isdir(f'{VOL}/40_tiempos'):
    objetivos.append(f'{VOL}/40_tiempos')
faltan = [c for c in BORRAR if not os.path.isdir(f'{CARGAS}/{c}')]
print('Se borraría:', *objetivos, sep='\n  ') if objetivos else print('No hay nada que borrar.')
if faltan:
    print('No están en 10_cargas (ya borradas o mal escritas):', faltan)
if CONFIRMAR_BORRADO == 'SI' and objetivos:
    for o in objetivos:
        if 'dbutils' in globals():
            dbutils.fs.rm(o, True)
        else:
            shutil.rmtree(o)
        print('borrado:', o)
elif objetivos:
    print("Para borrar escribir CONFIRMAR_BORRADO = 'SI' y volver a correr esta celda.")

## 6 · Estado final

In [ ]:
print('Cargas en las tablas:', cargas_en_tablas())
print('Carpetas en 10_cargas:', sorted(d for d in os.listdir(CARGAS) if os.path.isdir(f'{CARGAS}/{d}')) if os.path.isdir(CARGAS) else '(no existe)')
print('Hora (UTC):', ahora_utc())